In [ ]:
#4 simulated datasets
import neuroimage_analysis as na
import numpy as np
import matplotlib.pyplot as plt
import os 
import glob
import nibabel as nib
from tqdm import tqdm
from scipy.stats import  pearsonr
import pickle

# Directories

In [ ]:
dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
brain_template = nib.load(os.path.join(dir, "data/templates/Taylor_NHB_MNI152_T1_2mm_brain_mask_dil.nii.gz"))
brain_mask = brain_template.get_fdata() > 0

# Ground truth Schaefer 300 FC maps
schaefer_dir = os.path.join(dir, "data/schaefer_300_fcmap")
schaefer_files = sorted(glob.glob(os.path.join(schaefer_dir, '*.nii.gz')))
print(f'Found {len(schaefer_files)} Schaefer region FC maps for ground truth')

# Simulated lesion FC maps
lesion_dir = os.path.join(dir, "data/sim_lesions")
lesion_files = sorted(glob.glob(os.path.join(lesion_dir, 'lesion*AvgR.nii.gz')))
print(f'Found {len(lesion_files)} simulated lesion FC maps')

outdir = os.path.join(dir, "results")

pca_dir = '/Users/sasinm3/neuroimage_scripts/projects/arxiv/slnm_validation/data/templates/pca'

pc1_map = na.nifti_getdata(os.path.join(pca_dir, 'pca_voxelwise_pc1.nii.gz'))
pc2_map = na.nifti_getdata(os.path.join(pca_dir, 'pca_voxelwise_pc2.nii.gz'))
pc3_map = na.nifti_getdata(os.path.join(pca_dir, 'pca_voxelwise_pc3.nii.gz'))


# Simulated Datasets

Here, each simulated ‘study’ compared sLNM maps calculated from two separately simulated datasets with distinct ground truth symptom networks.

**Simulated lesions**
Lesions are 4-mm radius spheres placed at randomly sampled brain voxels. A pool of 500 lesion FC maps is generated in advance; for each simulated dataset, 100 subjects are bootstrapped (with replacement) from this pool. 

**Modeling clinical symptoms**
A ground truth network is defined by randomly selecting one of 300 Schaefer parcels; its whole-brain FC map defines the ground truth network, which represents the "true" disease network for their respective dataset. Symptoms ($S$) are modeled as a linear function of the Fisher z-transformed spatial correlation between the lesion FC map and the ground truth network ($R$), plus noise ($\epsilon$):

$$S = \theta R + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

The symptom variable $S$ was subsequently Z-scored. The effect size parameter $\eta^2$ controls the proportion of symptom variance explained by the ground truth network:

$$\eta^2 = \frac{\theta^2}{\theta^2 + \sigma^2}$$

We test four levels of effect size: $\eta^2 = 0.0$, $0.3$, $0.6$, and $0.99$.

**Convergence test (symptoms-permutation test)**
For each pair of simulated datasets, we compute sLNM maps and test whether their spatial similarity exceeds a permutation-based null distribution. For control analyses, we instead computed the partial correlation between sLNM maps controlling for connectome PC1, PC2, or PC3 as the test statistic.

In [ ]:
def partial_cor(A, B, C):

    r_AB = pearsonr(A, B)[0]
    r_BC = pearsonr(B, C)[0]
    r_AC = pearsonr(A, C)[0]

    r_AB_C_corrected = (r_AB - (r_AC*r_BC))/(np.sqrt(1 - r_AC**2)*np.sqrt(1 - r_BC **2))

    return r_AB_C_corrected  

def permute_network(A_fcmaps, B_fcmaps, A_outcomes, B_outcomes, n_permutations=1000):
    """
    Generate permuted sLNM maps for two datasets.

    For each permutation, behavioral outcomes are randomly shuffled before
    computing the sLNM map. The first permutation always uses the real outcomes.

    Returns
    -------
    dict with 'A_permuted' and 'B_permuted', each shaped (n_permutations x voxels)
    """
    def make_permutation_matrix(outcomes, n_perms):
        # First column is the real outcome; remaining columns are shuffled
        n = len(outcomes)
        matrix = np.zeros((n, n_perms))
        matrix[:, 0] = outcomes
        for i in range(1, n_perms):
            matrix[:, i] = np.random.permutation(outcomes)
        return matrix

    A_perm_matrix = make_permutation_matrix(A_outcomes, n_permutations)
    B_perm_matrix = make_permutation_matrix(B_outcomes, n_permutations)

    A_permuted_maps = na.voxel_outcome_correlation(A_fcmaps, A_perm_matrix)
    B_permuted_maps = na.voxel_outcome_correlation(B_fcmaps, B_perm_matrix)

    return {
        'A_permuted': A_permuted_maps,
        'B_permuted': B_permuted_maps
    }


def permute_network(A_fcmaps, B_fcmaps, A_outcomes, B_outcomes, n_permutations=1000):
    """
    Generate permuted sLNM maps for two datasets.

    For each permutation, behavioral outcomes are randomly shuffled before
    computing the sLNM map. The first permutation always uses the real outcomes.

    Returns
    -------
    dict with 'A_permuted' and 'B_permuted', each shaped (n_permutations x voxels)
    """
    def make_permutation_matrix(outcomes, n_perms):
        n = len(outcomes)
        matrix = np.zeros((n, n_perms))
        matrix[:, 0] = outcomes
        for i in range(1, n_perms):
            matrix[:, i] = np.random.permutation(outcomes)
        return matrix

    A_perm_matrix = make_permutation_matrix(A_outcomes, n_permutations)
    B_perm_matrix = make_permutation_matrix(B_outcomes, n_permutations)

    A_permuted_maps = na.voxel_outcome_correlation(A_fcmaps, A_perm_matrix)
    B_permuted_maps = na.voxel_outcome_correlation(B_fcmaps, B_perm_matrix)

    return {
        'A_permuted': A_permuted_maps,
        'B_permuted': B_permuted_maps
    }

def convergence_test(dataset_A, dataset_B, n_permutations=1000):
    A_fcmaps      = dataset_A['subject_maps']
    B_fcmaps      = dataset_B['subject_maps']
    A_scores      = dataset_A['scores']
    B_scores      = dataset_B['scores']
    A_ground_truth = dataset_A['ground_truth']
    B_ground_truth = dataset_B['ground_truth']

    ground_r = pearsonr(A_ground_truth, B_ground_truth).statistic

    A_slnm = na.voxel_outcome_correlation(A_fcmaps, A_scores[:, None]).flatten()
    B_slnm = na.voxel_outcome_correlation(B_fcmaps, B_scores[:, None]).flatten()

    emp_r = pearsonr(A_slnm, B_slnm).statistic

    permuted = permute_network(A_fcmaps, B_fcmaps, A_scores, B_scores, n_permutations)
    A_perm = permuted['A_permuted']  # (n_perms, voxels)
    B_perm = permuted['B_permuted']

    # Vectorized r(A_perm, B_perm) for all permutations
    permuted_r = na.pearson_rows(A_perm, B_perm) if hasattr(na, 'pearson_rows') else np.array([
        pearsonr(A_perm[i], B_perm[i]).statistic for i in range(n_permutations)
    ])

    # Full p-values
    p_value = (np.sum(permuted_r >= emp_r) + 1) / (n_permutations + 1)
    p_value_abs = (np.sum(np.abs(permuted_r) >= np.abs(emp_r)) + 1) / (n_permutations + 1)

    # Vectorized partial correlations for each PC
    # r(A_perm, PC) and r(B_perm, PC) computed once per PC, not per permutation
    result = {
        'ground_truth_r': ground_r,
        'empirical_r': emp_r,
        'permuted_r': permuted_r,
        'p_value': p_value,
        'p_value_abs': p_value_abs,
    }

    for pc_name, pc_map in [('pc1', pc1_map), ('pc2', pc2_map), ('pc3', pc3_map)]:
        # Vectorized: correlate all permuted maps with pc_map at once
        r_A_pc = na.pearson_rows(A_perm, pc_map)   # (n_perms,)
        r_B_pc = na.pearson_rows(B_perm, pc_map)   # (n_perms,)

        # Vectorized partial correlation: (r_AB - r_AC*r_BC) / (sqrt(1-r_AC^2)*sqrt(1-r_BC^2))
        perm_partial = (permuted_r - r_A_pc * r_B_pc) / (np.sqrt(1 - r_A_pc**2) * np.sqrt(1 - r_B_pc**2))

        # Empirical partial
        r_Aslnm_pc = pearsonr(A_slnm, pc_map)[0]
        r_Bslnm_pc = pearsonr(B_slnm, pc_map)[0]
        partial_r = (emp_r - r_Aslnm_pc * r_Bslnm_pc) / (np.sqrt(1 - r_Aslnm_pc**2) * np.sqrt(1 - r_Bslnm_pc**2))

        p_partial = (np.sum(perm_partial >= partial_r) + 1) / (n_permutations + 1)
        p_partial_abs = (np.sum(np.abs(perm_partial) >= np.abs(partial_r)) + 1) / (n_permutations + 1)

        gt_A = pearsonr(pc_map, A_ground_truth)[0]
        gt_B = pearsonr(pc_map, B_ground_truth)[0]

        result[f'{pc_name}_partial_r'] = partial_r
        result[f'{pc_name}_partial_p'] = p_partial
        result[f'{pc_name}_partial_p_abs'] = p_partial_abs
        result[f'{pc_name}_gt'] = [gt_A, gt_B]
        result[f'{pc_name}_slnm'] = [r_Aslnm_pc, r_Bslnm_pc]

    return result


In [ ]:
# test code on a single simulated sLNM study

np.random.seed(1)

dataset_A = na.gen_dataset(
    subject_maps=lesion_files,
    ground_truth_maps=schaefer_files,
    sample_size=100,
    effect_size=0.99,
    ground_truth_seed=17,
    z_transform = True
)

dataset_B = na.gen_dataset(
    subject_maps=lesion_files,
    ground_truth_maps=schaefer_files,
    sample_size=100,
    effect_size=0.99,
    ground_truth_seed=43,
    z_transform= True
)

print(f"Dataset A ground truth: Schaefer region {dataset_A['ground_truth_seed']}")
print(f"Dataset B ground truth: Schaefer region {dataset_B['ground_truth_seed']}")

result = convergence_test(dataset_A, dataset_B, n_permutations=1000)

print(f"\nGround truth similarity:    {result['ground_truth_r']:.3f}")
print(f"Empirical sLNM similarity:  {result['empirical_r']:.3f}")

print(f"\n{'':30s} {'Signed':>10s} {'Absolute':>10s}")
print(f"{'─'*52}")
print(f"{'Full correlation p-value':30s} {result['p_value']:10.3f} {result['p_value_abs']:10.3f}")

for pc in ['pc1', 'pc2', 'pc3']:
    print(f"\n--- {pc.upper()} ---")
    print(f"  Partial r:                {result[f'{pc}_partial_r']:.3f}")
    print(f"  Partial p (signed):       {result[f'{pc}_partial_p']:.3f}")
    print(f"  Partial p (absolute):     {result[f'{pc}_partial_p_abs']:.3f}")
    print(f"  {pc.upper()} — GT:   A={result[f'{pc}_gt'][0]:.3f}, B={result[f'{pc}_gt'][1]:.3f}")
    print(f"  {pc.upper()} — sLNM: A={result[f'{pc}_slnm'][0]:.3f}, B={result[f'{pc}_slnm'][1]:.3f}")

# Mass Simulations

In [ ]:
effect_sizes = [0.0,0.3,0.6,0.99] # or any effect size of interest 

for effect_size in effect_sizes: 

   # ── Tunable parameters ────────────────────────────────────────────────────────
    n_runs      = 1000
    n_perms     = 1000
    sample_size = 100

    # ─────────────────────────────────────────────────────────────────────────────

    np.random.seed(1)

    run_results = {
        'ground_truth_similarity': [],
        'slnm_similarity': [],
        'p_values': [],
        'is_significant': [],
        'p_values_abs': [],
        'is_significant_abs': [],
    }

    for pc in ['pc1', 'pc2', 'pc3']:
        run_results[f'{pc}_partial_similarity'] = []
        run_results[f'{pc}_p_values_partial'] = []
        run_results[f'{pc}_is_significant_partial'] = []
        run_results[f'{pc}_p_values_partial_abs'] = []
        run_results[f'{pc}_is_significant_partial_abs'] = []
        run_results[f'{pc}_gt'] = []
        run_results[f'{pc}_slnm'] = []

    for run in tqdm(range(n_runs), desc='Running simulations...'):
        seed_A, seed_B = np.random.choice(len(schaefer_files), size=2, replace=False)

        dataset_A = na.gen_dataset(
            subject_maps=lesion_files,
            ground_truth_maps=schaefer_files,
            sample_size=sample_size,
            effect_size=effect_size,
            ground_truth_seed=seed_A
        )
        dataset_B = na.gen_dataset(
            subject_maps=lesion_files,
            ground_truth_maps=schaefer_files,
            sample_size=sample_size,
            effect_size=effect_size,
            ground_truth_seed=seed_B
        )

        result = convergence_test(dataset_A, dataset_B, n_permutations=n_perms)

        run_results['ground_truth_similarity'].append(result['ground_truth_r'])
        run_results['slnm_similarity'].append(result['empirical_r'])
        run_results['p_values'].append(result['p_value'])
        run_results['is_significant'].append(result['p_value'] < 0.05)
        run_results['p_values_abs'].append(result['p_value_abs'])
        run_results['is_significant_abs'].append(result['p_value_abs'] < 0.05)

        for pc in ['pc1', 'pc2', 'pc3']:
            run_results[f'{pc}_partial_similarity'].append(result[f'{pc}_partial_r'])
            run_results[f'{pc}_p_values_partial'].append(result[f'{pc}_partial_p'])
            run_results[f'{pc}_is_significant_partial'].append(result[f'{pc}_partial_p'] < 0.05)
            run_results[f'{pc}_p_values_partial_abs'].append(result[f'{pc}_partial_p_abs'])
            run_results[f'{pc}_is_significant_partial_abs'].append(result[f'{pc}_partial_p_abs'] < 0.05)
            run_results[f'{pc}_gt'].append(result[f'{pc}_gt'])
            run_results[f'{pc}_slnm'].append(result[f'{pc}_slnm'])

    import pickle

    pkl_path = os.path.join(outdir, f'eta_{effect_size}_{n_runs}runs_results.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump(run_results, f)

    print(f'Saved: {pkl_path}')
    print(f'\n{"Test":<35s} {"Sig Rate":>10s}')
    print(f'{"─"*46}')
    print(f'{"Full (signed)":<35s} {np.mean(run_results["is_significant"]):>10.2%}')
    print(f'{"Full (absolute)":<35s} {np.mean(run_results["is_significant_abs"]):>10.2%}')
    for pc in ['pc1', 'pc2', 'pc3']:
        print(f'{f"Partial {pc.upper()} (signed)":<35s} {np.mean(run_results[f"{pc}_is_significant_partial"]):>10.2%}')
        print(f'{f"Partial {pc.upper()} (absolute)":<35s} {np.mean(run_results[f"{pc}_is_significant_partial_abs"]):>10.2%}')

# Validating the Test for Random Datasets (effect size = 0.0)

In [ ]:
from scipy.stats import kstest

# results from effect size = 0.0 studies: should produce significance rate ~5% and uniform p-value distribution 


zero_effect_pkl = os.path.join(outdir,'eta_0.0_1000runs_results.pkl')
with open(zero_effect_pkl, 'rb') as f:
    zero_effect_results = pickle.load(f)

sig_rate = np.mean(zero_effect_results['is_significant']) * 100
ks_stat, ks_p = kstest(zero_effect_results['p_values'], 'uniform')
print(f'Significance rate: {sig_rate:.1f}%, KS stat = {ks_stat:.3f}, KS p = {ks_p:.3f}')

# Comparing Meaningful Effect Sizes (0.3,0.6,0.99) 

In [ ]:
# first check the p-value is non-uniform (signal detected by the symptom-permutation test)
effect_sizes = [0.3, 0.6, 0.99]
n_runs = 1000

all_data = {}
for effect_size in effect_sizes:
    pkl_path = os.path.join(outdir, f'eta_{effect_size}_{n_runs}runs_results.pkl')
    with open(pkl_path, 'rb') as f:
        all_data[effect_size] = pickle.load(f)

for effect in effect_sizes:
    p_full = np.array(all_data[effect]['p_values'])
    ks_full, _ = kstest(p_full, 'uniform')
    print(f'η² = {effect} | KS stat = {ks_stat:.4f}, p = {(_)}')

# Plot Figure

In [ ]:
import pickle

effect_sizes = [0.3, 0.6, 0.99]
n_runs = 1000

color_full    = '#1A5DAD'  # strong blue
color_partial = '#2CA02C'  # vivid green
all_data = {}
for effect_size in effect_sizes:
    pkl_path = os.path.join(outdir, f'eta_{effect_size}_{n_runs}runs_results.pkl') # results presented in the manuscript
    with open(pkl_path, 'rb') as f:
        all_data[effect_size] = pickle.load(f)

bins        = np.arange(-1.0, 1.2, 0.2)
bin_centers = (bins[:-1] + bins[1:]) / 2
plt.rcParams['font.family'] = 'Arial'

fig, axes = plt.subplots(2, 3, figsize=(24, 14))

for col, effect in enumerate(effect_sizes):
    ground_r = np.array(all_data[effect]['ground_truth_similarity'])
    is_sig_full = np.array(all_data[effect]['is_significant'])
    is_sig_partial = np.array(all_data[effect]['pc1_is_significant_partial'])
    p_full = np.array(all_data[effect]['p_values'])
    p_partial = np.array(all_data[effect]['pc1_p_values_partial'])

    bin_indices = np.digitize(ground_r, bins)
    sig_full, sig_partial = [], []
    for b in range(1, len(bins)):
        in_bin = bin_indices == b
        n = np.sum(in_bin)
        sig_full.append(np.sum(is_sig_full[in_bin]) / n if n > 0 else np.nan)
        sig_partial.append(np.sum(is_sig_partial[in_bin]) / n if n > 0 else np.nan)
    sig_full = np.array(sig_full)
    sig_partial = np.array(sig_partial)

    # Row 0: Significance rate line plot
    ax = axes[0, col]
    ax.set_title(f'Study Effect Size = {effect}', fontsize = 25)
    ax.plot(bin_centers, sig_full, color=color_full, linewidth=5, marker='o', markersize=12, label = 'r')
    ax.plot(bin_centers, sig_partial, color=color_partial, linewidth=5, marker='o', markersize=12, label ='partial r$_{PC1}$')
    ax.axhline(0.05, color='red', linestyle='--', linewidth=2.5, label = '$\\alpha$ = 0.05')
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Significant Rate ($p < 0.05$)', fontsize = 22)
    ax.set_xlabel('Ground Truth Spatial r', fontsize = 22)
    ax.set_xticks(np.arange(-1.0, 1.1, 0.2))
    ax.tick_params(axis='both', labelsize=22)
    ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize = 22, loc= 'upper left')

    # Row 1: QQ plot
    ax = axes[1, col]
    exp_p = np.linspace(0, 1, len(p_full))
    ax.scatter(exp_p, np.sort(p_full), alpha=0.6, s=35, color=color_full, label = 'r')
    ax.scatter(exp_p, np.sort(p_partial), alpha=0.6, s=35, color=color_partial, label ='partial r$_{PC1}$')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.6, linewidth=2, label = 'uniform')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='both', labelsize=22)
    ax.set_ylabel('Observed p-value', fontsize = 22)
    ax.set_xlabel('Expected p-value', fontsize = 22)
    ax.grid(alpha=0.25)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize = 22, loc= 'upper left')

plt.tight_layout()
plt.show()

bins        = np.arange(-1.0, 1.2, 0.2)
for effect in effect_sizes:
    d = all_data[effect]
    print(f'\nη² = {effect}:')
    print(f'  Full (signed):        {np.mean(d["is_significant"]):.2%}')
    print(f'  Partial PC1 (signed): {np.mean(d["pc1_is_significant_partial"]):.2%}')




    ground_r    = np.array(all_data[effect]['ground_truth_similarity'])
    is_sig_full = np.array(all_data[effect]['is_significant'])
    is_sig_partial = np.array(all_data[effect]['pc1_is_significant_partial'])
    bin_indices = np.digitize(ground_r, bins)

    print(f'\nη² = {effect}:')
    print(f'{"bin":>14} | {"n":>6} | {"full sig%":>10} | {"partial sig%":>12}')
    print('-' * 52)
    for b in range(1, len(bins)):
        in_bin = bin_indices == b
        n = np.sum(in_bin)
        sf = np.sum(is_sig_full[in_bin]) / n if n > 0 else 0
        sp = np.sum(is_sig_partial[in_bin]) / n if n > 0 else 0
        print(f'{bins[b-1]:>5.1f}–{bins[b]:<5.1f}  | {n:>6} | {sf:>9.1%} | {sp:>11.1%}')

In [ ]:
import pickle

effect_sizes = [0.3, 0.6, 0.99]
n_runs = 1000

color_full    = '#1A5DAD'  # strong blue
color_partial = '#2CA02C'  # vivid green
all_data = {}
for effect_size in effect_sizes:
    pkl_path = os.path.join(outdir, f'eta_{effect_size}_{n_runs}runs_results.pkl')
    with open(pkl_path, 'rb') as f:
        all_data[effect_size] = pickle.load(f)

bins        = np.arange(-1.0, 1.2, 0.2)
bin_centers = (bins[:-1] + bins[1:]) / 2
plt.rcParams['font.family'] = 'Arial'

fig, axes = plt.subplots(4, 3, figsize=(24, 22))

for pc_idx, pc in enumerate([2, 3]):
    sig_row = pc_idx * 2
    qq_row  = pc_idx * 2 + 1
    for col, effect in enumerate(effect_sizes):
        ground_r = np.array(all_data[effect]['ground_truth_similarity'])
        is_sig_full = np.array(all_data[effect]['is_significant'])
        is_sig_partial = np.array(all_data[effect][f'pc{pc}_is_significant_partial'])
        p_full = np.array(all_data[effect]['p_values'])
        p_partial = np.array(all_data[effect][f'pc{pc}_p_values_partial'])

        bin_indices = np.digitize(ground_r, bins)
        sig_full, sig_partial = [], []
        for b in range(1, len(bins)):
            in_bin = bin_indices == b
            n = np.sum(in_bin)
            sig_full.append(np.sum(is_sig_full[in_bin]) / n if n > 0 else np.nan)
            sig_partial.append(np.sum(is_sig_partial[in_bin]) / n if n > 0 else np.nan)
        sig_full = np.array(sig_full)
        sig_partial = np.array(sig_partial)

        # Significance rate line plot
        ax = axes[sig_row, col]
        if sig_row == 0:
            ax.set_title(f'Study Effect Size = {effect}', fontsize=25)
        ax.plot(bin_centers, sig_full, color=color_full, linewidth=5, marker='o', markersize=12, label='r')
        ax.plot(bin_centers, sig_partial, color=color_partial, linewidth=5, marker='o', markersize=12, label=f'partial r$_{{PC{pc}}}$')
        ax.axhline(0.05, color='red', linestyle='--', linewidth=2.5, label='$\\alpha$ = 0.05')
        ax.set_ylim(0, 1.1)
        ax.set_ylabel(f'PC{pc}\nSignificant Rate ($p < 0.05$)', fontsize=22)
        ax.set_xlabel('Ground Truth Spatial r', fontsize=22)
        ax.set_xticks(np.arange(-1.0, 1.1, 0.2))
        ax.tick_params(axis='both', labelsize=19)
        ax.grid(alpha=0.25)
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(fontsize=20, loc='upper left')

        # QQ plot
        ax = axes[qq_row, col]
        exp_p = np.linspace(0, 1, len(p_full))
        ax.scatter(exp_p, np.sort(p_full), alpha=0.6, s=35, color=color_full, label='r')
        ax.scatter(exp_p, np.sort(p_partial), alpha=0.6, s=35, color=color_partial, label=f'partial r$_{{PC{pc}}}$')
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.6, linewidth=2, label='uniform')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.tick_params(axis='both', labelsize=19)
        ax.set_ylabel(f'PC{pc}\nObserved p-value', fontsize=22)
        ax.set_xlabel('Expected p-value', fontsize=22)
        ax.grid(alpha=0.25)
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(fontsize=20, loc='upper left')

plt.tight_layout()
plt.show()

for pc in [2, 3]:
    for effect in effect_sizes:
        d = all_data[effect]
        print(f'\nPC{pc}, η² = {effect}:')
        print(f'  Full (signed):          {np.mean(d["is_significant"]):.2%}')
        print(f'  Partial PC{pc} (signed): {np.mean(d[f"pc{pc}_is_significant_partial"]):.2%}')